# Train with Differential Sharpe Reward

This notebook shows how to train agents using the **Differential Sharpe Ratio** reward.

This is your original implementation based on Moody & Saffell (2001).

## What is Differential Sharpe?

- Implements proper online Sharpe ratio maximization
- Uses exponential decay for online statistics
- Theoretically grounded approach that fixes EMA variance issues
- Formula: `A_t = (R_t * σ_t - 0.5 * μ_t * (R_t - μ_t)) / σ_t²`

## Setup

In [1]:
from train import train, train_all_algos, train_both_agents
from utils import print_results, compare_results, save_results
from config import get_config

print("✓ Imports ready!")

✓ Imports ready!


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## 1. Train Technical Agent

Train technical agent with Differential Sharpe reward using PPO.

In [2]:
# Train with PPO
result_tech = train(
    agent_type='technical',
    algorithm='PPO',
    reward_type='differential_sharpe',
    verbose=True
)


Training: TECHNICAL - DIFFERENTIAL_SHARPE - PPO
Gamma: 0.9
Softmax temp: 1.0
Patience: 15
Environment: 201 dates, 7 assets, 20 features
Transaction cost: 25.0 bps
Using cpu device

Training for 300,000 steps...
-----------------------------
| time/              |      |
|    fps             | 1533 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 1437        |
|    iterations           | 2           |
|    time_elapsed         | 2           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009234047 |
|    clip_fraction        | 0.102       |
|    clip_range           | 0.2         |
|    entropy_loss         | -9.93       |
|    explained_variance   | 0.00509     |
|    learning_rate        | 0.0003      |
|    loss                 | 62.1

## 2. Train Sentiment Agent

In [5]:
# Train with PPO
result_sent = train(
    agent_type='sentiment',
    algorithm='PPO',
    reward_type='differential_sharpe',
    verbose=True
)


Training: SENTIMENT - DIFFERENTIAL_SHARPE - PPO
Gamma: 0.9
Softmax temp: 1.0
Patience: 15
Environment: 196 dates, 7 assets, 12 features
Transaction cost: 25.0 bps
Using cpu device

Training for 300,000 steps...
-----------------------------
| time/              |      |
|    fps             | 1390 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 1330        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009676876 |
|    clip_fraction        | 0.0917      |
|    clip_range           | 0.2         |
|    entropy_loss         | -9.93       |
|    explained_variance   | -0.00611    |
|    learning_rate        | 0.0003      |
|    loss                 | 75.9

## 3. Try All Algorithms

Test PPO, SAC, and A2C to find the best.

In [6]:
# Technical agent - all algorithms
print("\n" + "="*70)
print("Training TECHNICAL agent with all algorithms")
print("="*70)

tech_results = train_all_algos(
    agent_type='technical',
    reward_type='differential_sharpe',
    verbose=True
)


Training TECHNICAL agent with all algorithms

Training TECHNICAL with ALL algorithms
Reward: DIFFERENTIAL_SHARPE

PPO: Test Sharpe = 1.065

SAC: Test Sharpe = 0.622

A2C: Test Sharpe = 0.771

COMPARISON TABLE
agent_type algorithm         reward_type  train_sharpe  val_sharpe  test_sharpe  gamma  softmax_temperature  transaction_cost
 technical       PPO differential_sharpe      4.519365    2.270892     1.065184    0.9                  1.0            0.0025
 technical       A2C differential_sharpe      0.917736    2.485142     0.770646    0.9                  1.0            0.0025
 technical       SAC differential_sharpe      2.291160    2.259083     0.621785    0.9                  1.0            0.0025


🏆 Best: PPO (Test Sharpe: 1.065)


In [8]:
# Sentiment agent - all algorithms
print("\n" + "="*70)
print("Training SENTIMENT agent with all algorithms")
print("="*70)

sent_results = train_all_algos(
    agent_type='sentiment',
    reward_type='differential_sharpe',
    verbose=True
)


Training SENTIMENT agent with all algorithms

Training SENTIMENT with ALL algorithms
Reward: DIFFERENTIAL_SHARPE

PPO: Test Sharpe = 1.038

SAC: Test Sharpe = 0.969

A2C: Test Sharpe = 0.714

COMPARISON TABLE
agent_type algorithm         reward_type  train_sharpe  val_sharpe  test_sharpe  gamma  softmax_temperature  transaction_cost
 sentiment       PPO differential_sharpe      0.434777    2.277601     1.037639    0.9                  1.0            0.0025
 sentiment       SAC differential_sharpe      4.733974    2.368978     0.968893    0.9                  1.0            0.0025
 sentiment       A2C differential_sharpe      2.783725    2.621239     0.713983    0.9                  1.0            0.0025


🏆 Best: PPO (Test Sharpe: 1.038)


## 4. Compare Results

In [9]:
# Combine all results
all_results = tech_results + sent_results

# Show comparison
df = compare_results(all_results)

# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

best_tech = max(tech_results, key=lambda x: x['test_sharpe'])
best_sent = max(sent_results, key=lambda x: x['test_sharpe'])

print(f"\nBest Technical: {best_tech['algorithm']} - Test Sharpe = {best_tech['test_sharpe']:.3f}")
print(f"Best Sentiment: {best_sent['algorithm']} - Test Sharpe = {best_sent['test_sharpe']:.3f}")


COMPARISON TABLE
agent_type algorithm         reward_type  train_sharpe  val_sharpe  test_sharpe  gamma  softmax_temperature  transaction_cost
 technical       PPO differential_sharpe      4.519365    2.270892     1.065184    0.9                  1.0            0.0025
 sentiment       PPO differential_sharpe      0.434777    2.277601     1.037639    0.9                  1.0            0.0025
 sentiment       SAC differential_sharpe      4.733974    2.368978     0.968893    0.9                  1.0            0.0025
 technical       A2C differential_sharpe      0.917736    2.485142     0.770646    0.9                  1.0            0.0025
 sentiment       A2C differential_sharpe      2.783725    2.621239     0.713983    0.9                  1.0            0.0025
 technical       SAC differential_sharpe      2.291160    2.259083     0.621785    0.9                  1.0            0.0025


SUMMARY

Best Technical: PPO - Test Sharpe = 1.065
Best Sentiment: PPO - Test Sharpe = 1.038


## 5. Customize Settings

Change the decay factor (controls memory).

In [ ]:
# Get default config
config = get_config('differential_sharpe')

# Try different decay factors
# 0.95 = ~20 step memory (default)
# 0.90 = ~10 step memory (faster adaptation)
# 0.98 = ~50 step memory (slower adaptation)

config['decay_factor'] = 0.90

print(f"Training with decay_factor = {config['decay_factor']}")

# Train with custom config
result_custom = train(
    agent_type='technical',
    algorithm='PPO',
    reward_type='differential_sharpe',
    config=config,
    verbose=True
)

## 7. Save Results

In [10]:
# Save all results
results_dict = {
    'approach': 'differential_sharpe',
    'technical_results': tech_results,
    'sentiment_results': sent_results,
    'best_technical': best_tech,
    'best_sentiment': best_sent,
}

save_results(results_dict, 'results/differential_sharpe_results.json')

✓ Results saved to: results/differential_sharpe_results.json


## 📊 Understanding Differential Sharpe

The Differential Sharpe reward implements online Sharpe ratio maximization:

### Formula (from rewards.py):
```python
# Update online statistics with exponential decay
self._n_effective = 1.0 + self.decay * self._n_effective
alpha = 1.0 / max(self._n_effective, 1.0)

delta = port_return - self._mean_return
self._mean_return += alpha * delta
self._var_return = self.decay * self._var_return + (1.0 - self.decay) * delta**2

# Differential Sharpe Ratio formula
std_return = sqrt(self._var_return)
numerator = port_return * std_return - 0.5 * self._mean_return * delta
denominator = std_return**2
diff_sharpe = numerator / denominator

reward = diff_sharpe * sqrt(52)  # Annualize
```

### Key Parameter:
- **decay_factor** (default: 0.95)
  - 0.95 = ~20 step memory (balanced)
  - 0.90 = ~10 step memory (faster adaptation to regime changes)
  - 0.98 = ~50 step memory (more stable, slower adaptation)

### When to Use:
- ✅ Theoretically grounded (based on Moody & Saffell 2001)
- ✅ Fixes variance estimation issues in EMA approach
- ✅ Good for online Sharpe maximization
- ✅ Works well with changing market conditions

### Reference:
Moody, J., & Saffell, M. (2001). "Learning to Trade via Direct Reinforcement". IEEE Transactions on Neural Networks.

## 🎯 Next Steps

1. Try different decay factors (0.90, 0.95, 0.98)
2. Compare with EMA Sharpe and Multi-Objective
3. Test on different market conditions
4. Analyze the stability of returns